# 01 — Data exploration

**Owner:** Agent A.  
**Reads:** `data/processed/norman_hvg.h5ad` (output of `scripts/train_vae.py`'s preprocess step).  
**Writes:** nothing — visualization only.

**Sacred rule (CLAUDE.md §3 #8):** this notebook may visualize, but it may NOT define metrics. Every metric used here is imported from `src.analysis.metrics` or `src.analysis.latent_space`.

Sections:
1. Perturbation count distribution
2. HVG expression distributions
3. Control vs perturbed cell QC stats
4. Combinatorial-perturbation prevalence

In [ ]:
# Boilerplate — same in every notebook
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from hydra import compose, initialize_config_dir
with initialize_config_dir(version_base=None, config_dir=str(REPO_ROOT / "config")):
    cfg = compose(config_name="default")
print(cfg.paths.norman_processed_h5ad)

In [ ]:
import anndata
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

adata = anndata.read_h5ad(cfg.paths.norman_processed_h5ad)
print(f"shape: {adata.shape}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obs columns: {list(adata.obs.columns)}")

## 1. Perturbation count distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Count cells per perturbation, sorted descending
pert_counts = adata.obs["perturbation"].value_counts().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of cells-per-perturbation
axes[0].hist(pert_counts.values, bins=40, edgecolor="white", color="steelblue")
axes[0].set_xlabel("Cells per perturbation")
axes[0].set_ylabel("Count (perturbations)")
axes[0].set_title(f"Perturbation size distribution (n={len(pert_counts)} perturbations)")
axes[0].axvline(pert_counts.values.mean(), color="red", linestyle="--",
                label=f"mean={pert_counts.values.mean():.0f}")
axes[0].legend()

# Right: top-30 perturbations by cell count
top30 = pert_counts.head(30)
axes[1].barh(range(len(top30)), top30.values[::-1], color="steelblue")
axes[1].set_yticks(range(len(top30)))
axes[1].set_yticklabels(top30.index[::-1], fontsize=7)
axes[1].set_xlabel("Cell count")
axes[1].set_title("Top-30 perturbations by cell count")

plt.tight_layout()
plt.savefig("../artifacts/eval/figures/perturbation_counts.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total cells: {adata.n_obs:,}")
print(f"Unique perturbations: {len(pert_counts)}")
print(f"Control cells: {(adata.obs["perturbation"] == adata.uns.get("ctrl_label", "control")).sum():,}")


## 2. HVG expression distributions

In [ ]:
import scipy.sparse as sp

# Per-HVG mean and variance from normalized X (log1p)
X = adata.X
if sp.issparse(X):
    X_dense = X.toarray()
else:
    X_dense = np.asarray(X)

hvg_mean = X_dense.mean(axis=0)
hvg_var  = X_dense.var(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.log1p(hvg_mean), bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("log1p(mean expression)")
axes[0].set_ylabel("HVG count")
axes[0].set_title(f"HVG mean expression (n={len(hvg_mean)} HVGs)")

axes[1].scatter(np.log1p(hvg_mean), np.log1p(hvg_var), s=2, alpha=0.4, color="steelblue")
axes[1].set_xlabel("log1p(mean)")
axes[1].set_ylabel("log1p(variance)")
axes[1].set_title("Mean-variance relationship (log1p scale)")

plt.tight_layout()
plt.savefig("../artifacts/eval/figures/hvg_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"HVG mean range: [{hvg_mean.min():.3f}, {hvg_mean.max():.3f}]")
print(f"HVG variance range: [{hvg_var.min():.3f}, {hvg_var.max():.3f}]")


## 3. Control vs perturbed cell QC stats

In [ ]:
import seaborn as sns

# Compute per-cell QC metrics from raw counts layer
counts = adata.layers["counts"]
if sp.issparse(counts):
    total_counts = np.asarray(counts.sum(axis=1)).ravel()
    n_genes_detected = np.asarray((counts > 0).sum(axis=1)).ravel()
else:
    total_counts = counts.sum(axis=1)
    n_genes_detected = (counts > 0).sum(axis=1)

ctrl_label = adata.uns.get("ctrl_label", "control")
is_ctrl = adata.obs["perturbation"].values == ctrl_label

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for arr, title, ax in [
    (total_counts,    "Total UMI counts per cell",    axes[0]),
    (n_genes_detected, "Genes detected per cell",      axes[1]),
]:
    ax.hist(arr[is_ctrl],  bins=40, alpha=0.6, label=f"control (n={is_ctrl.sum():,})",  density=True)
    ax.hist(arr[~is_ctrl], bins=40, alpha=0.6, label=f"perturbed (n={(~is_ctrl).sum():,})", density=True)
    ax.set_xlabel(title)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../artifacts/eval/figures/qc_ctrl_vs_pert.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Ctrl   — mean counts: {total_counts[is_ctrl].mean():.0f}  mean genes: {n_genes_detected[is_ctrl].mean():.0f}")
print(f"Perturbed — mean counts: {total_counts[~is_ctrl].mean():.0f}  mean genes: {n_genes_detected[~is_ctrl].mean():.0f}")


## 4. Combinatorial perturbation prevalence

In [ ]:
import json

# Single vs combo counts
nperts = adata.obs.get("nperts", None)
if nperts is None and "nperts" in adata.obs.columns:
    nperts = adata.obs["nperts"]

if nperts is not None:
    n_ctrl   = (nperts == 0).sum()
    n_single = (nperts == 1).sum()
    n_combo  = (nperts == 2).sum()
else:
    n_single_genes = adata.uns["noop_idx"]
    pert_idx = adata.obs["perturbation_idx"].values
    n_ctrl   = (pert_idx == 0).sum()
    n_single = ((pert_idx >= 1) & (pert_idx <= n_single_genes)).sum()
    n_combo  = (pert_idx > n_single_genes).sum()

fig, ax = plt.subplots(figsize=(6, 4))
labels_pie = ["Control", "Single-gene", "Combo"]
sizes_pie  = [n_ctrl, n_single, n_combo]
colors_pie = ["#95a5a6", "#3498db", "#e67e22"]
wedges, texts, pcts = ax.pie(
    sizes_pie, labels=labels_pie, colors=colors_pie,
    autopct="%1.1f%%", startangle=90, pctdistance=0.8,
)
ax.set_title(f"Cell composition (total={sum(sizes_pie):,})")
plt.tight_layout()
plt.savefig("../artifacts/eval/figures/perturbation_types.png", dpi=150, bbox_inches="tight")
plt.show()

# Load and summarise held-out combos from pairs metadata (if available)
meta_path = "../artifacts/pairs/metadata.json"
try:
    meta = json.loads(open(meta_path).read())
    print(f"Held-out combos: {meta.get("held_out_combos", [])}")
except FileNotFoundError:
    print("pairs/metadata.json not yet generated — run make pairs first.")
print(f"Control: {n_ctrl:,}  Single: {n_single:,}  Combo: {n_combo:,}")
print(f"Combo fraction: {n_combo / (n_ctrl + n_single + n_combo):.1%}")
